In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
spark = SparkSession.builder\
        .appName("Estudo de Caso - Compras - VIVO")\
        .master("spark://spark-master:7077")\
        .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/09 15:57:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
#leitura do arquivo no hdfs
df = spark.read.csv("hdfs://namenode:8020/vivo/estudo_caso_compras.csv", sep=";", header=True, inferSchema=True)

# Analise Exploratória

In [4]:
df.show(5)

+-----------+-------------+--------------------+--------------+----------+------------------+--------------+-------+------------------+
|data_pedido|id_fornecedor|  categoria_material|valor_unitario|quantidade|prazo_entrega_dias|nota_qualidade|urgente|responsavel_compra|
+-----------+-------------+--------------------+--------------+----------+------------------+--------------+-------+------------------+
| 01/05/2023|         F007|Componentes eletr...|        478.74|        77|                23|             5|    não|           Daniela|
| 14/03/2023|         F020|          Embalagens|        181.47|         4|                 3|             4|    não|               Ana|
| 05/10/2023|         F015|          Embalagens|        439.51|        62|                19|             3|    não|            Carlos|
| 18/08/2023|         F011|         Ferramentas|        404.52|        65|                 8|             2|    não|            Carlos|
| 14/05/2023|         F008|         Ferramentas|

In [5]:
df.describe().show()

[Stage 3:>                                                          (0 + 1) / 1]

+-------+-----------+-------------+--------------------+-----------------+-----------------+------------------+------------------+-------+------------------+
|summary|data_pedido|id_fornecedor|  categoria_material|   valor_unitario|       quantidade|prazo_entrega_dias|    nota_qualidade|urgente|responsavel_compra|
+-------+-----------+-------------+--------------------+-----------------+-----------------+------------------+------------------+-------+------------------+
|  count|        200|          200|                 200|              200|              200|               200|               200|    200|               200|
|   mean|       NULL|         NULL|                NULL|         539.1267|           50.645|             13.22|              2.91|   NULL|              NULL|
| stddev|       NULL|         NULL|                NULL|308.3052620115818|29.28715867605771| 8.581363469459554|1.4325364930849407|   NULL|              NULL|
|    min| 01/02/2023|         F001|Componentes eletr

In [6]:
#vendo tipo das variáveis
df.printSchema()

root
 |-- data_pedido: string (nullable = true)
 |-- id_fornecedor: string (nullable = true)
 |-- categoria_material: string (nullable = true)
 |-- valor_unitario: double (nullable = true)
 |-- quantidade: integer (nullable = true)
 |-- prazo_entrega_dias: integer (nullable = true)
 |-- nota_qualidade: integer (nullable = true)
 |-- urgente: string (nullable = true)
 |-- responsavel_compra: string (nullable = true)



In [7]:
#conversao data pedido de string para tipo date
df = df.withColumn("data_pedido", F.to_date("data_pedido", "dd/MM/yyyy"))

In [8]:
#agrupamento de colunas não numéricas para verificação de nulos e quais tipos de dados temos
df.groupBy("categoria_material")\
    .agg(F.count("categoria_material").alias("qtd"))\
    .show(truncate=False)

df.groupBy("urgente")\
    .agg(F.count("urgente").alias("qtd"))\
    .show(truncate=False)

df.groupBy("responsavel_compra")\
    .agg(F.count("responsavel_compra").alias("qtd"))\
    .show(truncate=False)

+-----------------------+---+
|categoria_material     |qtd|
+-----------------------+---+
|Componentes eletrônicos|39 |
|Embalagens             |35 |
|Ferramentas            |43 |
|Matéria-prima          |45 |
|Serviços               |38 |
+-----------------------+---+

+-------+---+
|urgente|qtd|
+-------+---+
|sim    |35 |
|não    |165|
+-------+---+

+------------------+---+
|responsavel_compra|qtd|
+------------------+---+
|Bruno             |40 |
|Daniela           |44 |
|Carlos            |38 |
|Ana               |42 |
|Eduardo           |36 |
+------------------+---+



In [9]:
df.select(F.countDistinct("id_fornecedor")).show()

+-----------------------------+
|count(DISTINCT id_fornecedor)|
+-----------------------------+
|                           20|
+-----------------------------+



In [10]:
#calculo da coluna total, vlr unitario x qtd
df = df.withColumn("total", F.col("valor_unitario") * F.col("quantidade"))

#substituição de sim e não por 1 e 0 na coluna urgente, deixando-a binária
df = df.withColumn("urgente", F.when(F.col("urgente")=="sim", 1).otherwise(0))

In [11]:
df.printSchema()

root
 |-- data_pedido: date (nullable = true)
 |-- id_fornecedor: string (nullable = true)
 |-- categoria_material: string (nullable = true)
 |-- valor_unitario: double (nullable = true)
 |-- quantidade: integer (nullable = true)
 |-- prazo_entrega_dias: integer (nullable = true)
 |-- nota_qualidade: integer (nullable = true)
 |-- urgente: integer (nullable = false)
 |-- responsavel_compra: string (nullable = true)
 |-- total: double (nullable = true)



In [12]:
#df final com tratamentos basicos
df.show()

+-----------+-------------+--------------------+--------------+----------+------------------+--------------+-------+------------------+------------------+
|data_pedido|id_fornecedor|  categoria_material|valor_unitario|quantidade|prazo_entrega_dias|nota_qualidade|urgente|responsavel_compra|             total|
+-----------+-------------+--------------------+--------------+----------+------------------+--------------+-------+------------------+------------------+
| 2023-05-01|         F007|Componentes eletr...|        478.74|        77|                23|             5|      0|           Daniela|          36862.98|
| 2023-03-14|         F020|          Embalagens|        181.47|         4|                 3|             4|      0|               Ana|            725.88|
| 2023-10-05|         F015|          Embalagens|        439.51|        62|                19|             3|      0|            Carlos|          27249.62|
| 2023-08-18|         F011|         Ferramentas|        404.52|       

# Diagnóstico

In [13]:
df_diag = df

In [14]:
#Fornecedor com maior custo e melhor custo beneficio

total_geral = df_diag.select("total")\
    .agg(F.sum("total").alias("total_geral"))

fornecedor = df_diag.groupBy("id_fornecedor") \
    .agg(
    F.sum("total").alias("total"),\
    (F.sum(F.col("nota_qualidade") * F.col("quantidade")) / F.sum("quantidade")).alias("avg_qual"),\
    F.sum("quantidade").alias("quantidade"),\
    )\
    .crossJoin(total_geral)\
    .withColumn("% total", 100 * F.col("total") / F.col("total_geral"))\
    .withColumn("avg_un", F.col("total") / F.col("quantidade"))\
    .withColumn("custo_beneficio", F.col("avg_un") / F.col("avg_qual"))\
    .withColumn("total", F.round("total", 2)) \
    .withColumn("% total", F.round("% total", 2))\
    .withColumn("avg_un", F.round("avg_un", 2))\
    .withColumn("avg_qual", F.round("avg_qual", 2))\
    .withColumn("custo_beneficio", F.round("custo_beneficio", 2))

            
fornecedor.select("id_fornecedor", "total", "% total", "custo_beneficio")\
    .orderBy(F.desc("total"))\
    .show(5)

fornecedor.select("id_fornecedor", "total", "avg_qual", "custo_beneficio")\
    .orderBy(F.asc("custo_beneficio"))\
    .show(5)

+-------------+---------+-------+---------------+
|id_fornecedor|    total|% total|custo_beneficio|
+-------------+---------+-------+---------------+
|         F015| 461364.9|   8.57|         233.37|
|         F003| 422869.5|   7.86|         227.96|
|         F012|346828.35|   6.44|         211.61|
|         F016|332957.01|   6.19|         368.72|
|         F011|307856.62|   5.72|         194.85|
+-------------+---------+-------+---------------+
only showing top 5 rows

+-------------+---------+--------+---------------+
|id_fornecedor|    total|avg_qual|custo_beneficio|
+-------------+---------+--------+---------------+
|         F018| 89645.16|    3.11|          80.83|
|         F019|213031.33|    3.28|         122.93|
|         F007|292941.18|    3.12|         136.38|
|         F005|191870.01|    2.35|         140.36|
|         F009|293442.36|    3.57|         145.34|
+-------------+---------+--------+---------------+
only showing top 5 rows



In [15]:
#urgencia vs aumento de custo

In [16]:
df_diag.groupBy("urgente") \
    .agg(
        F.count("*").alias("qtd_pedidos"),
        F.avg("total").alias("media_custo"),
        F.sum("total").alias("custo_total")
    ) \
    .show()

+-------+-----------+------------------+------------------+
|urgente|qtd_pedidos|       media_custo|       custo_total|
+-------+-----------+------------------+------------------+
|      1|         35|29762.839142857145|1041699.3700000001|
|      0|        165| 26307.74127272727|        4340777.31|
+-------+-----------+------------------+------------------+



In [17]:
#categorias com maior impacto no orçamento
df_diag.groupBy("categoria_material") \
  .agg(F.sum("total").alias("total")) \
  .orderBy(F.desc("total")) \
  .show(truncate=False)

+-----------------------+------------------+
|categoria_material     |total             |
+-----------------------+------------------+
|Serviços               |1340234.34        |
|Ferramentas            |1317397.48        |
|Matéria-prima          |1092419.9300000002|
|Componentes eletrônicos|909974.1299999998 |
|Embalagens             |722450.7999999999 |
+-----------------------+------------------+



# Modelagem e Otimização

In [18]:
df_ml = df
df_ml.show()

+-----------+-------------+--------------------+--------------+----------+------------------+--------------+-------+------------------+------------------+
|data_pedido|id_fornecedor|  categoria_material|valor_unitario|quantidade|prazo_entrega_dias|nota_qualidade|urgente|responsavel_compra|             total|
+-----------+-------------+--------------------+--------------+----------+------------------+--------------+-------+------------------+------------------+
| 2023-05-01|         F007|Componentes eletr...|        478.74|        77|                23|             5|      0|           Daniela|          36862.98|
| 2023-03-14|         F020|          Embalagens|        181.47|         4|                 3|             4|      0|               Ana|            725.88|
| 2023-10-05|         F015|          Embalagens|        439.51|        62|                19|             3|      0|            Carlos|          27249.62|
| 2023-08-18|         F011|         Ferramentas|        404.52|       

In [19]:
#criação de variaveis dummies para colunas de categoria e responsavel de compra
colunas_dummies = ['categoria_material', 'responsavel_compra']

for c in colunas_dummies:
    categorias = [row[c] for row in df_ml.select(c).distinct().collect()]
    for cat in categorias:
        df_ml = df_ml.withColumn(
            f"{c}_{cat}", 
            F.when(F.col(c) == cat, 1).otherwise(0)
        )

#retirando colunas de categoria e respoinsavel (ja foram criadas dummies), data de pedido (irrelevante para analise), id_fornecedor(devido a pequena amostra 200 linhas, e 20 fornecedores, não é ideal)
df_ml = df_ml.drop(*colunas_dummies, 'data_pedido', 'id_fornecedor','valor_unitario')

#renomando coluna total para modelo de regressao linear
df_ml = df_ml.withColumnRenamed("total", "label")

#df final para aplicar regressao linear
df_ml.show()

+----------+------------------+--------------+-------+------------------+------------------------------------------+-----------------------------+------------------------------+--------------------------------+---------------------------+------------------------+--------------------------+-------------------------+----------------------+--------------------------+
|quantidade|prazo_entrega_dias|nota_qualidade|urgente|             label|categoria_material_Componentes eletrônicos|categoria_material_Embalagens|categoria_material_Ferramentas|categoria_material_Matéria-prima|categoria_material_Serviços|responsavel_compra_Bruno|responsavel_compra_Daniela|responsavel_compra_Carlos|responsavel_compra_Ana|responsavel_compra_Eduardo|
+----------+------------------+--------------+-------+------------------+------------------------------------------+-----------------------------+------------------------------+--------------------------------+---------------------------+------------------------+---

In [20]:
#import de bibliotecas necessarias para regressao linear
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [21]:
feature_columns = [
     'quantidade',
     'prazo_entrega_dias',
     'nota_qualidade',
     'urgente',
     'categoria_material_Componentes eletrônicos',
     'categoria_material_Embalagens',
     'categoria_material_Ferramentas',
     'categoria_material_Matéria-prima',
     'categoria_material_Serviços',
     'responsavel_compra_Bruno',
     'responsavel_compra_Daniela',
     'responsavel_compra_Carlos',
     'responsavel_compra_Ana',
     'responsavel_compra_Eduardo'
]

#montando vetor para ml
assembler = VectorAssembler(inputCols = feature_columns, outputCol = 'features')
dataset_ml = assembler.transform(df_ml).select('features', 'label')

#quebra do df para 70% de treino e 30% de teste
treino, teste = dataset_ml.randomSplit([0.7, 0.3], seed=101)

In [22]:
#instancia do modelo de regressao
lr = LinearRegression()

#fitting do modelo para sample de treino
model = lr.fit(treino)

#aplicacao do modelo na base de teste
result = model.transform(teste)

evaluator = RegressionEvaluator()

#mérticas de root mean squared error e r2
print("R2: %f" % evaluator.evaluate(result, {evaluator.metricName: "r2"}))
print("RMSE: %f" % evaluator.evaluate(result, {evaluator.metricName: "rmse"}))

# Mostrar coeficientes e suas correlações
for i in range(len(feature_columns)):
    print(feature_columns[i], ":", model.coefficients[i])

R2: 0.268616
RMSE: 20158.102408
quantidade : 524.0751218217761
prazo_entrega_dias : 124.73794296511669
nota_qualidade : 977.9161514767505
urgente : -434.3284752400508
categoria_material_Componentes eletrônicos : -4306.624531053758
categoria_material_Embalagens : -2352.662929429547
categoria_material_Ferramentas : 5084.246007352676
categoria_material_Matéria-prima : -1322.5445017492882
categoria_material_Serviços : 2514.471155967961
responsavel_compra_Bruno : 2318.437564173091
responsavel_compra_Daniela : 3288.9443930975403
responsavel_compra_Carlos : -5452.707342750644
responsavel_compra_Ana : 1917.1578942059166
responsavel_compra_Eduardo : -3329.2700273659802


In [23]:
result.select("features", "label", "prediction").show(10, truncate=False)


+-----------------------------------------------+-----------------+------------------+
|features                                       |label            |prediction        |
+-----------------------------------------------+-----------------+------------------+
|(14,[0,1,2,3,5,10],[21.0,8.0,5.0,1.0,1.0,1.0]) |13062.84         |12488.274708746798|
|(14,[0,1,2,3,5,10],[79.0,13.0,4.0,1.0,1.0,1.0])|63892.83         |42530.40533775864 |
|(14,[0,1,2,3,6,11],[39.0,7.0,4.0,1.0,1.0,1.0]) |18293.34         |19514.230008030936|
|(14,[0,1,2,3,6,12],[44.0,25.0,5.0,1.0,1.0,1.0])|43399.4          |32727.66997894523 |
|(14,[0,1,2,3,7,9],[34.0,5.0,3.0,1.0,1.0,1.0])  |20987.52         |17030.81675933684 |
|(14,[0,1,2,3,7,10],[34.0,11.0,1.0,1.0,1.0,1.0])|22511.74         |16793.91894309849 |
|(14,[0,1,2,3,7,11],[87.0,21.0,2.0,1.0,1.0,1.0])|86394.48         |38053.54424493236 |
|(14,[0,1,2,3,7,13],[1.0,28.0,2.0,1.0,1.0,1.0]) |986.77           |-4020.313315599905|
|(14,[0,1,2,3,8,13],[50.0,12.0,3.0,1.0,1.0,

In [24]:
rfr = RandomForestRegressor(seed=42)


grid = ParamGridBuilder() \
    .addGrid(rfr.numTrees, [10, 20, 30]) \
    .addGrid(rfr.maxDepth, [5, 10]) \
    .addGrid(rfr.maxBins, [10, 32, 45]) \
    .build()


evaluator = RegressionEvaluator()

rfr_cv = CrossValidator(
    estimator=rfr,
    estimatorParamMaps=grid,
    evaluator=evaluator,
    numFolds=3,
    seed=42
)

modelo_rfr_cv = rfr_cv.fit(treino)

In [25]:
modelo_teste = modelo_rfr_cv.transform(teste)
modelo_teste.select("features", "label", "prediction").show(10, truncate=False)
print('Teste')
print("R2: %f" % evaluator.evaluate(modelo_teste, {evaluator.metricName: "r2"}))
print("RMSE: %f" % evaluator.evaluate(modelo_teste, {evaluator.metricName: "rmse"}))

+-----------------------------------------------+-----------------+------------------+
|features                                       |label            |prediction        |
+-----------------------------------------------+-----------------+------------------+
|(14,[0,1,2,3,5,10],[21.0,8.0,5.0,1.0,1.0,1.0]) |13062.84         |6678.214925925927 |
|(14,[0,1,2,3,5,10],[79.0,13.0,4.0,1.0,1.0,1.0])|63892.83         |39833.8470857143  |
|(14,[0,1,2,3,6,11],[39.0,7.0,4.0,1.0,1.0,1.0]) |18293.34         |16593.578249999995|
|(14,[0,1,2,3,6,12],[44.0,25.0,5.0,1.0,1.0,1.0])|43399.4          |23771.400655555553|
|(14,[0,1,2,3,7,9],[34.0,5.0,3.0,1.0,1.0,1.0])  |20987.52         |21033.999285714286|
|(14,[0,1,2,3,7,10],[34.0,11.0,1.0,1.0,1.0,1.0])|22511.74         |14330.012952380954|
|(14,[0,1,2,3,7,11],[87.0,21.0,2.0,1.0,1.0,1.0])|86394.48         |38974.868355158724|
|(14,[0,1,2,3,7,13],[1.0,28.0,2.0,1.0,1.0,1.0]) |986.77           |10401.082833333332|
|(14,[0,1,2,3,8,13],[50.0,12.0,3.0,1.0,1.0,

# Teste regressão linear

In [26]:
new_input = [{
    "quantidade": 100,
    "prazo_entrega_dias": 29,
    "nota_qualidade": 5,
    "urgente": 1,
    "categoria_material_Componentes eletrônicos": 0,
    "categoria_material_Embalagens": 1,
    "categoria_material_Ferramentas": 0,
    "categoria_material_Matéria-prima": 0,
    "categoria_material_Serviços": 0,
    "responsavel_compra_Bruno": 0,
    "responsavel_compra_Daniela": 0,
    "responsavel_compra_Carlos": 1,
    "responsavel_compra_Ana": 0,
    "responsavel_compra_Eduardo": 0,
}]

# criar dataframe com a nova linha
df_input_teste = spark.createDataFrame(new_input)

input_vetor = assembler.transform(df_input_teste).select('features')

modelo_input = model.transform(input_vetor)

modelo_input.select("features", "prediction").show()

+--------------------+------------------+
|            features|        prediction|
+--------------------+------------------+
|(14,[0,1,2,3,5,11...|47768.054399086366|
+--------------------+------------------+

